# 04 — Error Analysis & Linguistic Insights

Deep analysis of model errors for Standard → Dialect (Feature C):
- Error taxonomy classification
- Dialect marker recall by region
- VnCoreNLP POS comparison
- Qualitative examples for the report

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path("..").resolve()))

sns.set_theme(style="whitegrid")

PRED_PATH = Path("../results/metrics/model_predictions.jsonl")

## Load Predictions

In [ ]:
def load_preds(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records

if PRED_PATH.exists():
    all_preds = load_preds(PRED_PATH)
    std2dia = [r for r in all_preds if r["task"].startswith("std2dialect")]
    dia2std = [r for r in all_preds if r["task"] == "dialect2std"]
    print(f"Total: {len(all_preds)}, Std2Dia: {len(std2dia)}, Dia2Std: {len(dia2std)}")
else:
    print(f"Predictions not found at {PRED_PATH}. Run inference first.")
    all_preds = []
    std2dia = []
    dia2std = []

## Automated Error Classification

In [ ]:
from src.evaluation.error_analysis import detect_error_type

if std2dia:
    error_types = [detect_error_type(r) for r in std2dia]
    error_counts = Counter(e for e in error_types if e is not None)
    correct = sum(1 for e in error_types if e is None)

    print(f"Correct: {correct}/{len(std2dia)} ({correct/len(std2dia)*100:.1f}%)")
    print(f"Error distribution:")
    for err_type, count in error_counts.most_common():
        print(f"  {err_type}: {count} ({count/len(std2dia)*100:.1f}%)")

    # Plot
    if error_counts:
        fig, ax = plt.subplots(figsize=(8, 5))
        types = list(error_counts.keys())
        counts = [error_counts[t] for t in types]
        ax.barh(types, counts, color=sns.color_palette("Reds_d", len(types)))
        ax.set_xlabel("Count")
        ax.set_title("Error Type Distribution (Standard → Dialect)")
        plt.tight_layout()
        plt.savefig("../results/figures/error_distribution.png", dpi=150)
        plt.show()

## Dialect Feature Recall by Region

In [ ]:
from src.evaluation.metrics import compute_dfr

if std2dia:
    preds = [r["prediction"] for r in std2dia]
    refs = [r["target"] for r in std2dia]
    regions = [r.get("region", "") for r in std2dia]

    dfr = compute_dfr(preds, refs, regions)
    print("Dialect Feature Recall:")
    for k, v in dfr.items():
        print(f"  {k}: {v:.4f}")

## Qualitative Examples for Report

Show good examples and bad examples side by side.

In [ ]:
if std2dia:
    # Good examples (prediction matches reference)
    good = [r for r, e in zip(std2dia, error_types) if e is None][:10]
    print("GOOD EXAMPLES (correct transfers):")
    print("=" * 60)
    for r in good:
        print(f"  [{r['region']}] SRC: {r['source']}")
        print(f"           PRD: {r['prediction']}")
        print(f"           REF: {r['target']}")
        print()

    # Bad examples (errors)
    bad = [(r, e) for r, e in zip(std2dia, error_types) if e is not None][:10]
    print("\nBAD EXAMPLES (errors):")
    print("=" * 60)
    for r, err in bad:
        print(f"  [{r['region']}] {err}")
        print(f"           SRC: {r['source']}")
        print(f"           PRD: {r['prediction']}")
        print(f"           REF: {r['target']}")
        print()

## Comparison: Model vs Baselines

In [ ]:
from src.evaluation.metrics import compute_bleu, compute_rouge_l

# Template — fill in with actual baseline prediction files
results_table = []

for name, path in [
    ("Model", PRED_PATH),
    ("LAI", Path("../results/metrics/lai_predictions.jsonl")),
    ("Rule-based", Path("../results/metrics/rule_predictions.jsonl")),
]:
    if not path.exists():
        continue
    preds_all = load_preds(path)
    s2d = [r for r in preds_all if r["task"].startswith("std2dialect")]
    if not s2d:
        continue
    p = [r["prediction"] for r in s2d]
    r = [r["target"] for r in s2d]
    results_table.append({
        "Method": name,
        "BLEU": f"{compute_bleu(p, r):.2f}",
        "ROUGE-L": f"{compute_rouge_l(p, r):.4f}",
        "Count": len(s2d),
    })

if results_table:
    df = pd.DataFrame(results_table)
    print(df.to_markdown(index=False))
else:
    print("No results available yet.")